In [1]:
import torch
import triton
import triton.language as tl
DEVICE = torch.device(f'cuda:{torch.cuda.current_device()}')
print(f"Device: {DEVICE}")

Device: cuda:0


In [ ]:
@triton.jit
def add_kernel(x_ptr, y_ptr, z_ptr, n_elements, BLOCK_SIZE: tl.constexpr):
    pid = tl.program_id(axis=0)
    offset = pid * BLOCK_SIZE + tl.arange(0, BLOCK_SIZE)
    mask = offset < n_elements
    x = tl.load(x_ptr + offset, mask=mask)
    y = tl.load(y_ptr + offset, mask=mask)
    tl.store(z_ptr + offset, x + y, mask=mask)

In [ ]:
BLOCK_SIZE = 512
def add(x:torch.Tensor, y:torch.Tensor):
  output = torch.empty_like(x)
  n_elements = x.numel()
  # safety checking 
  assert x.shape == y.shape, "Input tensors must have the same shape"
  assert x.device == y.device and x.device == DEVICE, "Input tensors must be on the same device as the kernel"
  grid = triton.cdiv(n_elements, BLOCK_SIZE)
  add_kernel[(grid,)](x, y, output, n_elements, BLOCK_SIZE=BLOCK_SIZE)
  return output

In [4]:
def test_correctness(size, atol=1e-4, rtol=1e-4, device=DEVICE):
  x = torch.randn(size, device=device)
  y = torch.randn(size, device=device)
  z_tri = add(x, y)
  z_ref = x + y
  torch.testing.assert_close(z_tri, z_ref, atol=atol, rtol=rtol)
  print("Correctness test passed!")

In [ ]:
# test config and perf_report
@triton.testing.perf_report(
  triton.testing.Benchmark(
    x_names=["size"],
    x_vals=[2**i for i in range(12, 28)],
    x_log=True,
    line_arg='provider',
    line_vals=['triton', 'torch'],
    line_names=['Triton', 'PyTorch'],
    styles=[('blue', '-'), ('green', '-')],
    ylabel='GB/s',
    plot_name="add_performance",
    args={},
  )
)
def benchmark(size, provider):
  x = torch.rand(size, device=DEVICE, dtype=torch.float32)
  y = torch.rand(size, device=DEVICE, dtype=torch.float32)
  quantiles = [0.5, 0.05, 0.95]
  if provider == 'triton':
    ms, min_ms, max_ms = triton.testing.do_bench(lambda: add(x, y), quantiles=quantiles)
  else:
    ms, min_ms, max_ms = triton.testing.do_bench(lambda: x + y, quantiles=quantiles)
  gbps = lambda ms: 3 * x.numel() * x.element_size() / (ms * 1e-3) / 1e9
  return gbps(ms), gbps(max_ms), gbps(min_ms)

In [ ]:
# benchmark.run(print_data=True, show_plots=True)